In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.datasets import load_gbsg2
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
import statsmodels.api as sm
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
from sklearn.preprocessing import StandardScaler

In [2]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [3]:
# Need to choose patient_id from OUS_D3 in response_OUS
data = list(OUS_D3['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D3 with response_OUS
clinical_train = pd.merge(OUS_D3, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS', 'LRC', 'event_LRC'])]

In [4]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

In [5]:
""" Takes too long time 
# Check collinearity for OUS data 
df = clinical_train

# Create a correlation matrix
correlation_matrix = df.corr()

plt.figure(figsize=(15, 12))

# Plot a heatmap
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)
plt.show()
""" 

" Takes too long time \n# Check collinearity for OUS data \ndf = clinical_train\n\n# Create a correlation matrix\ncorrelation_matrix = df.corr()\n\nplt.figure(figsize=(15, 12))\n\n# Plot a heatmap\nsns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)\nplt.show()\n"

## Test dataset: MAASTRO 

In [6]:
(MAASTRO_D3['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [7]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [8]:
# need to choose patient_id from MAASTRO_D3 in response_MAASTRO
data = list(MAASTRO_D3['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 
response_MAASTRO

,patient_id,OS,OS_event,LRC,LRC_event,DFS,DFS_event
0,1,62.43,0.0,62.43,0.0,62.43,0.0
1,2,60.00,0.0,60.00,0.0,60.00,0.0
2,3,44.43,1.0,8.83,1.0,8.83,1.0
3,4,37.20,1.0,19.37,0.0,19.73,1.0
5,6,59.23,0.0,59.23,0.0,59.23,0.0
...,...,...,...,...,...,...,...
109,110,19.00,1.0,13.27,1.0,13.27,1.0
110,111,85.87,1.0,82.83,0.0,85.87,1.0
111,112,42.87,0.0,42.87,0.0,42.87,0.0
112,113,58.93,0.0,58.93,0.0,58.93,0.0


In [9]:
# Merge MAASTRO_D3 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D3, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'DFS_event', 'LRC', 'LRC_event'])]
clinical_test

,patient_id,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,OS,OS_event
0,1,55,0,0,1,0,0,1,1,1,...,0.028808,0.837387,0.000026,0.001705,0.122209,0.000026,0.008291,0.000341,62.43,0.0
1,2,55,0,0,1,0,0,0,0,0,...,0.049615,0.806842,0.000167,0.002958,0.128976,0.000167,0.008148,0.000335,60.00,0.0
2,3,55,0,0,1,0,0,0,0,1,...,0.019514,0.830272,0.000057,0.001831,0.137282,0.000000,0.008641,0.000229,44.43,1.0
3,4,61,1,0,0,0,1,1,0,1,...,0.047155,0.760949,0.000000,0.003597,0.171595,0.000080,0.013187,0.000240,37.20,1.0
4,6,70,0,0,1,0,0,1,1,1,...,0.033990,0.824865,0.000000,0.002116,0.128134,0.000035,0.009379,0.000529,59.23,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,110,66,1,0,0,0,1,0,0,0,...,0.017489,0.834590,0.000000,0.001321,0.137194,0.000078,0.008706,0.000078,19.00,1.0
95,111,63,0,0,0,0,1,0,0,1,...,0.029979,0.821431,0.000000,0.000543,0.137620,0.000054,0.008907,0.000489,85.87,1.0
96,112,63,0,0,1,0,0,1,1,1,...,0.017354,0.859957,0.000000,0.000988,0.115099,0.000028,0.005700,0.000141,42.87,0.0
97,113,54,0,0,1,0,0,1,1,0,...,0.025399,0.847908,0.000000,0.001487,0.117654,0.000000,0.005759,0.000038,58.93,0.0


In [10]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [11]:
# Some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,OS,OS_event


In [12]:
# Set X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS'])]

# Set y 
y = clinical_train.loc[:, ['OS', 'event_OS']]

In [13]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [14]:
# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

X_train:  (139, 388)
y_train:  (139,)


In [15]:
# Change the name of a column 'OS_event' in the clincial_test 
clinical_test.rename(columns = {'OS_event' : 'event_OS'}, inplace = True)

In [16]:
# Set X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'event_OS'])]

# Set y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['OS', 'event_OS']]

# Change y_MAASTRO into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_OS'], y_MAASTRO['OS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

(99, 390)

## Feature Selection

### COX PLSR 

In [20]:
# Choose features from the result of Cox PLSR in R
selected_features = [
"uicc8_III-IV",
"cavum_oris",
"hpv_related",
"charlson",
"hypopharynx",
"shape_MajorAxisLength",
"LBP_120_PET"   
]

In [21]:
X_plsr = X.loc[:, selected_features]
X_new = X_plsr.copy()

In [22]:
# Selecct the columns from X_MAASTRO
MAASTRO_new = X_MAASTRO.loc[:, selected_features]

# Standardization

In [23]:
# Copy the original X for later 
original_X = X.copy()

In [24]:
categorical_columns = ['female', 
                        'cavum_oris',
                        'oropharynx',
                        'hypopharynx',
                        'larynx',
                        'histgrade_high',
                        'hpv_related',
                        'charlson',
                        'uicc8_III-IV']

# Standardize X_new, the new data with the selected features only 
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in X_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
X_new_categoric = X_new[columns_to_drop]
X_new_numeric = X_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
scaler = StandardScaler() 
X_new_numeric_columns = X_new_numeric.columns
X_new_numeric_index = X_new_numeric.index 
X_new_numeric_std = scaler.fit_transform(X_new_numeric)
X_new_numeric_std = pd.DataFrame(X_new_numeric_std,
                                 columns=X_new_numeric_columns, 
                                 index=X_new_numeric_index)
X_new_std = pd.concat([X_new_numeric_std, X_new_categoric], axis=1)

# Change the order of the X_new_std 
X_new_std = X_new_std[X_new.columns]

In [25]:
# Standardize X_MAASTRO 
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in MAASTRO_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
MAASTRO_new_categoric = MAASTRO_new[columns_to_drop]
MAASTRO_new_numeric = MAASTRO_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns

MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_index = MAASTRO_new_numeric.index 
MAASTRO_new_numeric_std = scaler.transform(MAASTRO_new_numeric)
MAASTRO_new_numeric_std = pd.DataFrame(MAASTRO_new_numeric_std,
                                 columns=MAASTRO_new_numeric_columns, 
                                 index=MAASTRO_new_numeric_index)
MAASTRO_new_std = pd.concat([MAASTRO_new_numeric_std, MAASTRO_new_categoric], axis=1)

# Change the order of the X_new_std 
MAASTRO_new = MAASTRO_new[X_new.columns]
MAASTRO_new_std = MAASTRO_new_std[X_new.columns]

In [26]:
X_new

,uicc8_III-IV,cavum_oris,hpv_related,charlson,hypopharynx,shape_MajorAxisLength,LBP_120_PET
0,0.0,0,0.0,0,0,42.073251,0.140311
1,0.0,0,0.0,1,0,24.613845,0.191058
2,1.0,1,0.0,1,0,48.030294,0.126531
3,0.0,0,0.0,1,0,25.589900,0.192388
4,0.0,0,0.0,1,0,34.684750,0.202073
...,...,...,...,...,...,...,...
134,0.0,0,1.0,0,0,33.069705,0.152626
135,1.0,0,1.0,0,0,41.043692,0.142778
136,0.0,0,1.0,1,0,36.618802,0.140582
137,1.0,0,1.0,1,0,45.870392,0.156640


In [27]:
X_new_std

,uicc8_III-IV,cavum_oris,hpv_related,charlson,hypopharynx,shape_MajorAxisLength,LBP_120_PET
0,0.0,0,0.0,0,0,-0.098422,-0.476493
1,0.0,0,0.0,1,0,-1.209119,1.084993
2,1.0,1,0.0,1,0,0.280542,-0.900502
3,0.0,0,0.0,1,0,-1.147026,1.125919
4,0.0,0,0.0,1,0,-0.568449,1.423898
...,...,...,...,...,...,...,...
134,0.0,0,1.0,0,0,-0.671191,-0.097562
135,1.0,0,1.0,0,0,-0.163918,-0.400566
136,0.0,0,1.0,1,0,-0.445412,-0.468144
137,1.0,0,1.0,1,0,0.143137,0.025938


In [28]:
MAASTRO_new 

,uicc8_III-IV,cavum_oris,hpv_related,charlson,hypopharynx,shape_MajorAxisLength,LBP_120_PET
0,0,0,1,1,0,50.002093,0.122209
1,1,0,0,0,0,41.753334,0.128976
2,1,0,0,1,0,44.375483,0.137282
3,1,0,0,1,0,46.115989,0.171595
4,0,0,1,1,0,54.394967,0.128134
...,...,...,...,...,...,...,...
94,1,0,0,0,0,34.218615,0.137194
95,1,0,0,1,0,51.046869,0.137620
96,1,0,1,1,0,50.417953,0.115099
97,0,0,1,0,0,44.901412,0.117654


In [29]:
MAASTRO_new_std

,uicc8_III-IV,cavum_oris,hpv_related,charlson,hypopharynx,shape_MajorAxisLength,LBP_120_PET
0,0,0,1,1,0,0.405980,-1.033478
1,1,0,0,0,0,-0.118774,-0.825249
2,1,0,0,1,0,0.048037,-0.569695
3,1,0,0,1,0,0.158761,0.486120
4,0,0,1,1,0,0.685437,-0.851180
...,...,...,...,...,...,...,...
94,1,0,0,0,0,-0.598102,-0.572399
95,1,0,0,1,0,0.472444,-0.559284
96,1,0,1,1,0,0.432435,-1.252248
97,0,0,1,0,0,0.081495,-1.173644


# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [30]:
# Setting the y format for skf below  
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-18 03:19:43,616] A new study created in memory with name: no-name-3d30eda4-3c50-425e-99ae-8c505c81cec0
python(7352) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.7662337662337663
Fold 2 C-index: 0.7678571428571429
Fold 3 C-index: 0.8480392156862745
Fold 4 C-index: 0.8059071729957806


[I 2024-04-18 03:19:51,799] A new study created in memory with name: no-name-efdd777f-1870-47bb-ae30-5097ad05994c


Fold 5 C-index: 0.704225352112676
[I 2024-04-18 03:19:51,644] Trial 0 finished with value: 0.7784525299771281 and parameters: {}. Best is trial 0 with value: 0.7784525299771281.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7784525299771281], datetime_start=datetime.datetime(2024, 4, 18, 3, 19, 43, 805461), datetime_complete=datetime.datetime(2024, 4, 18, 3, 19, 51, 642687), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7784525299771281


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.16762996230119587
Fold 2 IBS: 0.16471344311707362
Fold 3 IBS: 0.12271184148213712
Fold 4 IBS: 0.15606953244496122
Fold 5 IBS: 0.23145868866366215
[I 2024-04-18 03:19:53,146] Trial 0 finished with value: 0.168516693601806 and parameters: {}. Best is trial 0 with value: 0.168516693601806.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.168516693601806], datetime_start=datetime.datetime(2024, 4, 18, 3, 19, 51, 853954), datetime_complete=datetime.datetime(2024, 4, 18, 3, 19, 53, 145691), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.168516693601806


In [31]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

In [32]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.778
train_ibs:  0.169


#### Test

In [33]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [34]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

CoxPHSurvivalAnalysis()

Concordance index: 0.62
IBS score: 0.232


In [35]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [36]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

### 2. CoxnetSurvivalAnalysis - Ridge

#### Train

In [37]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-18 03:19:53,935] A new study created in memory with name: no-name-caa722ee-449e-4b9e-a709-2579d62d5ab8


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.5692640692640693
Fold 2 C-index: 0.7678571428571429
Fold 3 C-index: 0.7132352941176471
Fold 4 C-index: 0.6919831223628692
Fold 5 C-index: 0.6643192488262911
[I 2024-04-18 03:19:54,569] Trial 0 finished with value: 0.6813317754856039 and parameters: {}. Best is trial 0 with value: 0.6813317754856039.


[I 2024-04-18 03:19:54,644] A new study created in memory with name: no-name-619f2f12-db38-407b-848c-13bdfcc54a72




* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6813317754856039], datetime_start=datetime.datetime(2024, 4, 18, 3, 19, 54, 211271), datetime_complete=datetime.datetime(2024, 4, 18, 3, 19, 54, 564333), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6813317754856039


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.213976519404854
Fold 2 IBS: 0.22157790763480684
Fold 3 IBS: 0.20453594247607443
Fold 4 IBS: 0.22473803447097768
Fold 5 IBS: 0.21812430841953245
[I 2024-04-18 03:19:55,089] Trial 0 finished with value: 0.21659054248124904 and parameters: {}. Best is trial 0 with value: 0.21659054248124904.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.21659054248124904], datetime_start=datetime.datetime(2024, 4, 18, 3, 19, 54, 698874), datetime_complete=datetime.datetime(2024, 4, 18, 3, 19, 55, 88817), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.21659054248124904


In [38]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [39]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.681
train_ibs:  0.217


#### Test

In [40]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [41]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.578


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.221


In [42]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [43]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-18 03:19:55,641] A new study created in memory with name: no-name-0ed37a46-4fae-4f04-a0c1-1bad96c68fa1


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.7532467532467533
Fold 2 C-index: 0.7678571428571429
Fold 3 C-index: 0.8578431372549019
Fold 4 C-index: 0.8059071729957806
Fold 5 C-index: 0.7089201877934272


[I 2024-04-18 03:19:56,694] A new study created in memory with name: no-name-9ae24dd1-c760-4c8c-b002-ba2ea66da00e


[I 2024-04-18 03:19:56,633] Trial 0 finished with value: 0.7787548788296013 and parameters: {}. Best is trial 0 with value: 0.7787548788296013.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7787548788296013], datetime_start=datetime.datetime(2024, 4, 18, 3, 19, 55, 837424), datetime_complete=datetime.datetime(2024, 4, 18, 3, 19, 56, 628172), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7787548788296013


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.16849796886747778
Fold 2 IBS: 0.1659079675044365
Fold 3 IBS: 0.12315202765362111
Fold 4 IBS: 0.15530953920397786
Fold 5 IBS: 0.2301462993017957
[I 2024-04-18 03:19:58,210] Trial 0 finished with value: 0.1686027605062618 and parameters: {}. Best is trial 0 with value: 0.1686027605062618.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.1686027605062618], datetime_start=datetime.datetime(2024, 4, 18, 3, 19, 56, 878079), datetime_complete=datetime.datetime(2024, 4, 18, 3, 19, 58, 209721), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.1686027605062618


In [44]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [45]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.779
train_ibs:  0.169


#### Test

In [46]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [47]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.623


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.23


In [48]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [49]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-18 03:19:58,978] A new study created in memory with name: no-name-98610998-ad5a-44f7-95db-7003182888bc


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.7532467532467533
Fold 2 C-index: 0.7723214285714286
Fold 3 C-index: 0.8578431372549019
Fold 4 C-index: 0.8059071729957806
Fold 5 C-index: 0.7089201877934272
[I 2024-04-18 03:19:59,928] Trial 0 finished with value: 0.7796477359724584 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.7796477359724584.
Fold 1 C-index: 0.7532467532467533
Fold 2 C-index: 0.7678571428571429
Fold 3 C-index: 0.8529411764705882
Fold 4 C-index: 0.8059071729957806
Fold 5 C-index: 0.7089201877934272
[I 2024-04-18 03:20:00,877] Trial 1 finished with value: 0.7777744866727385 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.7796477359724584.
Fold 1 C-index: 0.7575757575757576
Fold 2 C-index: 0.7678571428571429
Fold 3 C-index: 0.8529411764705882
Fold 4 C-index: 0.8059071729957806
Fold 5 C-index: 0.7089201877934272
[I 2024-04-18 03:20:01,695] Trial 2 finished with value: 0.7786402875385393 and parameters: {'l1_ratio': 0.22692876841884668}.

Fold 1 C-index: 0.7532467532467533
Fold 2 C-index: 0.7723214285714286
Fold 3 C-index: 0.8578431372549019
Fold 4 C-index: 0.8059071729957806
Fold 5 C-index: 0.7089201877934272
[I 2024-04-18 03:20:27,667] Trial 24 finished with value: 0.7796477359724584 and parameters: {'l1_ratio': 0.7602371709740536}. Best is trial 0 with value: 0.7796477359724584.
Fold 1 C-index: 0.7532467532467533
Fold 2 C-index: 0.7678571428571429
Fold 3 C-index: 0.8578431372549019
Fold 4 C-index: 0.8059071729957806
Fold 5 C-index: 0.7089201877934272
[I 2024-04-18 03:20:29,429] Trial 25 finished with value: 0.7787548788296013 and parameters: {'l1_ratio': 0.891573855041803}. Best is trial 0 with value: 0.7796477359724584.
Fold 1 C-index: 0.7532467532467533
Fold 2 C-index: 0.7723214285714286
Fold 3 C-index: 0.8578431372549019
Fold 4 C-index: 0.8059071729957806
Fold 5 C-index: 0.7089201877934272
[I 2024-04-18 03:20:31,428] Trial 26 finished with value: 0.7796477359724584 and parameters: {'l1_ratio': 0.4514399534035586}.

Fold 1 C-index: 0.7532467532467533
Fold 2 C-index: 0.7678571428571429
Fold 3 C-index: 0.8578431372549019
Fold 4 C-index: 0.8059071729957806
Fold 5 C-index: 0.7089201877934272
[I 2024-04-18 03:21:04,969] Trial 48 finished with value: 0.7787548788296013 and parameters: {'l1_ratio': 0.40176254986805804}. Best is trial 0 with value: 0.7796477359724584.
Fold 1 C-index: 0.7532467532467533
Fold 2 C-index: 0.7723214285714286
Fold 3 C-index: 0.8578431372549019
Fold 4 C-index: 0.8059071729957806
Fold 5 C-index: 0.7089201877934272
[I 2024-04-18 03:21:06,151] Trial 49 finished with value: 0.7796477359724584 and parameters: {'l1_ratio': 0.8313598689405559}. Best is trial 0 with value: 0.7796477359724584.
Fold 1 C-index: 0.7532467532467533
Fold 2 C-index: 0.7723214285714286
Fold 3 C-index: 0.8578431372549019
Fold 4 C-index: 0.8059071729957806
Fold 5 C-index: 0.7089201877934272
[I 2024-04-18 03:21:07,357] Trial 50 finished with value: 0.7796477359724584 and parameters: {'l1_ratio': 0.7120258016641781

...

Fold 1 IBS: 0.16844173638213747
Fold 2 IBS: 0.16515999724040445
Fold 3 IBS: 0.1230674092004973
Fold 4 IBS: 0.15532968544182732
Fold 5 IBS: 0.22987092445957294
[I 2024-04-18 03:22:17,948] Trial 0 finished with value: 0.16837395054488788 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.16837395054488788.
Fold 1 IBS: 0.16840948858106042
Fold 2 IBS: 0.1637108842507164
Fold 3 IBS: 0.12290243936655076
Fold 4 IBS: 0.15552587898929446
Fold 5 IBS: 0.2290849133112781
[I 2024-04-18 03:22:19,800] Trial 1 finished with value: 0.16792672089978003 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.16792672089978003.
Fold 1 IBS: 0.16837583795884792
Fold 2 IBS: 0.1634444033419204
Fold 3 IBS: 0.12285330672189478
Fold 4 IBS: 0.15553401964446517
Fold 5 IBS: 0.2289892533206302
[I 2024-04-18 03:22:22,068] Trial 2 finished with value: 0.1678393641975517 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 2 with value: 0.167839364197551

Fold 1 IBS: 0.16843339228982424
Fold 2 IBS: 0.16406705615949221
Fold 3 IBS: 0.12293161703222383
Fold 4 IBS: 0.15547587210092845
Fold 5 IBS: 0.22941697359956037
[I 2024-04-18 03:23:01,903] Trial 25 finished with value: 0.1680649822364058 and parameters: {'l1_ratio': 0.3804525958179419}. Best is trial 18 with value: 0.16755261007764882.
Fold 1 IBS: 0.1684177900660573
Fold 2 IBS: 0.16384957460920266
Fold 3 IBS: 0.12291520963379016
Fold 4 IBS: 0.15547621667156794
Fold 5 IBS: 0.22933803752905746
[I 2024-04-18 03:23:03,325] Trial 26 finished with value: 0.1679993657019351 and parameters: {'l1_ratio': 0.3312072877416866}. Best is trial 18 with value: 0.16755261007764882.
Fold 1 IBS: 0.16827995765183623
Fold 2 IBS: 0.16276847565567504
Fold 3 IBS: 0.1227702439018671
Fold 4 IBS: 0.15559926529038748
Fold 5 IBS: 0.228353856203132
[I 2024-04-18 03:23:05,063] Trial 27 finished with value: 0.16755435974057958 and parameters: {'l1_ratio': 0.07046652721581821}. Best is trial 18 with value: 0.1675526100

Fold 5 IBS: 0.2288503907654318
[I 2024-04-18 03:23:51,034] Trial 49 finished with value: 0.16776538274348626 and parameters: {'l1_ratio': 0.1739865498896483}. Best is trial 46 with value: 0.16753371174261983.
Fold 1 IBS: 0.16823956884471547
Fold 2 IBS: 0.16265775015630018
Fold 3 IBS: 0.12276745593861005
Fold 4 IBS: 0.22201778171735595
Fold 5 IBS: 0.22826456696892697
[I 2024-04-18 03:23:52,897] Trial 50 finished with value: 0.18078942472518172 and parameters: {'l1_ratio': 0.040430207194092474}. Best is trial 46 with value: 0.16753371174261983.
Fold 1 IBS: 0.16827235241940472
Fold 2 IBS: 0.16256633572583526
Fold 3 IBS: 0.1227614155405021
Fold 4 IBS: 0.2219109704942683
Fold 5 IBS: 0.22837743103018027
[I 2024-04-18 03:23:55,180] Trial 51 finished with value: 0.18077770104203814 and parameters: {'l1_ratio': 0.04206766761484858}. Best is trial 46 with value: 0.16753371174261983.
Fold 1 IBS: 0.16829044602227222
Fold 2 IBS: 0.16286944216161936
Fold 3 IBS: 0.122799437559214
Fold 4 IBS: 0.155607

Fold 1 IBS: 0.16832861518122158
Fold 2 IBS: 0.1630316298309345
Fold 3 IBS: 0.12282007224328821
Fold 4 IBS: 0.15559108824572546
Fold 5 IBS: 0.22854773723235974
[I 2024-04-18 03:24:44,747] Trial 74 finished with value: 0.16766382854670592 and parameters: {'l1_ratio': 0.14538128014992321}. Best is trial 46 with value: 0.16753371174261983.
Fold 1 IBS: 0.16827616054249042
Fold 2 IBS: 0.16293598838646187
Fold 3 IBS: 0.1227955705921141
Fold 4 IBS: 0.15557344374701365
Fold 5 IBS: 0.2284861248858399
[I 2024-04-18 03:24:46,946] Trial 75 finished with value: 0.167613457630784 and parameters: {'l1_ratio': 0.11207878786036493}. Best is trial 46 with value: 0.16753371174261983.
Fold 1 IBS: 0.16843090599960045
Fold 2 IBS: 0.16385845830158044
Fold 3 IBS: 0.1229180903008412
Fold 4 IBS: 0.15551921419809686
Fold 5 IBS: 0.22924594837427384
[I 2024-04-18 03:24:48,997] Trial 76 finished with value: 0.16799452343487856 and parameters: {'l1_ratio': 0.32134083380611417}. Best is trial 46 with value: 0.16753371

Fold 4 IBS: 0.15559409221063014
Fold 5 IBS: 0.22870671398377174
[I 2024-04-18 03:25:34,590] Trial 98 finished with value: 0.16767815480757448 and parameters: {'l1_ratio': 0.13111375797605596}. Best is trial 89 with value: 0.16753328852706267.
Fold 1 IBS: 0.16831714561719655
Fold 2 IBS: 0.1632108507105827
Fold 3 IBS: 0.12282163960669765
Fold 4 IBS: 0.15556068379359772
Fold 5 IBS: 0.22877769998936398
[I 2024-04-18 03:25:36,681] Trial 99 finished with value: 0.16773760394348775 and parameters: {'l1_ratio': 0.16965192632929055}. Best is trial 89 with value: 0.16753328852706267.


* Best trial for IBS: 
 FrozenTrial(number=89, state=TrialState.COMPLETE, values=[0.16753328852706267], datetime_start=datetime.datetime(2024, 4, 18, 3, 25, 14, 653064), datetime_complete=datetime.datetime(2024, 4, 18, 3, 25, 16, 797317), params={'l1_ratio': 0.04922481027815988}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'l1_ratio': FloatDistribution(high=1.0, log=False, low=0.0001, st

In [50]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [51]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.78
train_ibs:  0.168


#### Test

In [52]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [53]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.6964995386793018)

test_cindex : 0.623


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.04922481027815988)

test_ibs:  0.229


In [54]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [55]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-18 03:25:37,580] A new study created in memory with name: no-name-f3282b91-65d2-495b-b373-83ef2f0c25fd


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.7467532467532467
Fold 2 C-index: 0.7098214285714286
Fold 3 C-index: 0.8235294117647058
Fold 4 C-index: 0.7974683544303798
Fold 5 C-index: 0.6854460093896714
[I 2024-04-18 03:25:52,637] Trial 0 finished with value: 0.7526036901818866 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.7526036901818866.
Fold 1 C-index: 0.7186147186147186
Fold 2 C-index: 0.8080357142857143
Fold 3 C-index: 0.8431372549019608
Fold 4 C-index: 0.8016877637130801
Fold 5 C-index: 0.6737089201877934
[I 2024-04-18 03:26:03,923] Trial 1 finished with value: 0.7690368743406535 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, 

Fold 3 C-index: 0.8382352941176471
Fold 4 C-index: 0.8396624472573839
Fold 5 C-index: 0.7417840375586855
[I 2024-04-18 03:28:58,539] Trial 15 finished with value: 0.7983302951806827 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 14, 'min_samples_leaf': 10, 'max_depth': 6, 'n_estimators': 369, 'oob_score': True, 'max_samples': 0.8284238710883769, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.17583629499904482, 'warm_start': True}. Best is trial 15 with value: 0.7983302951806827.
Fold 1 C-index: 0.7316017316017316
Fold 2 C-index: 0.8660714285714286
Fold 3 C-index: 0.8235294117647058
Fold 4 C-index: 0.8270042194092827
Fold 5 C-index: 0.7323943661971831
[I 2024-04-18 03:29:03,536] Trial 16 finished with value: 0.7961202315088663 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 15, 'min_samples_leaf': 16, 'max_depth': 7, 'n_estimators': 175, 'oob_score': True, 'max_samples': 0.8042262963180606, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.1904497006797

Fold 1 C-index: 0.70995670995671
Fold 2 C-index: 0.8705357142857143
Fold 3 C-index: 0.8578431372549019
Fold 4 C-index: 0.8354430379746836
Fold 5 C-index: 0.7464788732394366
[I 2024-04-18 03:31:29,001] Trial 30 finished with value: 0.8040514945422894 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 6, 'min_samples_leaf': 6, 'max_depth': 16, 'n_estimators': 325, 'oob_score': True, 'max_samples': 0.9958344170463831, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.20001380215328324, 'warm_start': True}. Best is trial 28 with value: 0.8155652756086147.
Fold 1 C-index: 0.7316017316017316
Fold 2 C-index: 0.8660714285714286
Fold 3 C-index: 0.8725490196078431
Fold 4 C-index: 0.8607594936708861
Fold 5 C-index: 0.7605633802816901
[I 2024-04-18 03:31:37,377] Trial 31 finished with value: 0.8183090107467159 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 8, 'min_samples_leaf': 9, 'max_depth': 16, 'n_estimators': 362, 'oob_score': True, 'max_samples': 0.9452517320699256, '

...

Fold 5 IBS: 0.20606070459603618
[I 2024-04-18 03:40:35,649] Trial 31 finished with value: 0.17142647822250986 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 19, 'min_samples_leaf': 4, 'max_depth': 6, 'n_estimators': 141, 'oob_score': False, 'max_samples': 0.7191779197966834, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.0953426015021738}. Best is trial 28 with value: 0.16990612676953618.
Fold 1 IBS: 0.18250138852812883
Fold 2 IBS: 0.17113429637227343
Fold 3 IBS: 0.14623517642012188
Fold 4 IBS: 0.1363752556022476
Fold 5 IBS: 0.20606526324852095
[I 2024-04-18 03:40:41,859] Trial 32 finished with value: 0.16846227603425853 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 19, 'min_samples_leaf': 2, 'max_depth': 4, 'n_estimators': 177, 'oob_score': False, 'max_samples': 0.7002017603582902, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.037741041052811355}. Best is trial 32 with value: 0.16846227603425853.
Fold 1 IBS: 0.18202492847751467
Fold 2 IBS: 0.173

Fold 1 IBS: 0.2140121718199979
Fold 2 IBS: 0.22149935338683438
Fold 3 IBS: 0.2047666373465207
Fold 4 IBS: 0.22488209831535308
Fold 5 IBS: 0.2183757411797355
[I 2024-04-18 03:41:47,185] Trial 47 finished with value: 0.21670720040968833 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 9, 'min_samples_leaf': 3, 'max_depth': 15, 'n_estimators': 130, 'oob_score': True, 'max_samples': 0.5234170058690094, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.38143282767559583}. Best is trial 36 with value: 0.1681090932517048.
Fold 1 IBS: 0.2139972127669122
Fold 2 IBS: 0.22124358236933428
Fold 3 IBS: 0.2047527974187217
Fold 4 IBS: 0.22464662730656376
Fold 5 IBS: 0.2185790125333401
[I 2024-04-18 03:41:50,388] Trial 48 finished with value: 0.2166438464789744 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 6, 'min_samples_leaf': 1, 'max_depth': 12, 'n_estimators': 88, 'oob_score': False, 'max_samples': 0.6960402324901204, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.

Fold 1 IBS: 0.1766842532379171
Fold 2 IBS: 0.1698257997704228
Fold 3 IBS: 0.15680662914587148
Fold 4 IBS: 0.14176388782718463
Fold 5 IBS: 0.20453883734596576
[I 2024-04-18 03:42:48,352] Trial 63 finished with value: 0.16992388146547238 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 17, 'min_samples_leaf': 6, 'max_depth': 9, 'n_estimators': 82, 'oob_score': True, 'max_samples': 0.7967542684659779, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.06784168011112049}. Best is trial 36 with value: 0.1681090932517048.
Fold 1 IBS: 0.18432304609942296
Fold 2 IBS: 0.17230962226867122
Fold 3 IBS: 0.15160216224521647
Fold 4 IBS: 0.1391825722296759
Fold 5 IBS: 0.2017956551731704
[I 2024-04-18 03:42:51,873] Trial 64 finished with value: 0.16984261160323139 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 15, 'min_samples_leaf': 5, 'max_depth': 11, 'n_estimators': 69, 'oob_score': True, 'max_samples': 0.8456585073090606, 'max_features': 'auto', 'min_weight_fraction_leaf'

Fold 1 IBS: 0.20024801689348162
Fold 2 IBS: 0.2057130041741153
Fold 3 IBS: 0.1962060834371484
Fold 4 IBS: 0.21051437171900578
Fold 5 IBS: 0.20833522870797178
[I 2024-04-18 03:44:16,812] Trial 79 finished with value: 0.20420334098634457 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 2, 'min_samples_leaf': 6, 'max_depth': 11, 'n_estimators': 41, 'oob_score': False, 'max_samples': 0.12074719353470553, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.04303701256971175}. Best is trial 75 with value: 0.16731372484760992.
Fold 1 IBS: 0.22003650915905537
Fold 2 IBS: 0.17780026433656795
Fold 3 IBS: 0.17858928964906395
Fold 4 IBS: 0.1469680858174885
Fold 5 IBS: 0.20607478690239545
[I 2024-04-18 03:44:26,806] Trial 80 finished with value: 0.18589378717291424 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 19, 'min_samples_leaf': 1, 'max_depth': 12, 'n_estimators': 199, 'oob_score': False, 'max_samples': 0.9999277178523378, 'max_features': None, 'min_weight_fraction_lea

Fold 1 IBS: 0.19197606811810808
Fold 2 IBS: 0.15832469275331668
Fold 3 IBS: 0.15508398644550617
Fold 4 IBS: 0.17886741180476176
Fold 5 IBS: 0.1922677833948585
[I 2024-04-18 03:45:07,889] Trial 95 finished with value: 0.17530398850331025 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 18, 'min_samples_leaf': 3, 'max_depth': 12, 'n_estimators': 4, 'oob_score': True, 'max_samples': 0.785839145990297, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.10722424812745346}. Best is trial 75 with value: 0.16731372484760992.
Fold 1 IBS: 0.17318005344396062
Fold 2 IBS: 0.18079748347172456
Fold 3 IBS: 0.14761255568940362
Fold 4 IBS: 0.13827071021934526
Fold 5 IBS: 0.20015023689231282
[I 2024-04-18 03:45:12,765] Trial 96 finished with value: 0.16800220794334936 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 18, 'min_samples_leaf': 1, 'max_depth': 20, 'n_estimators': 92, 'oob_score': False, 'max_samples': 0.5947494725419651, 'max_features': 'auto', 'min_weight_fraction_lea

In [56]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [57]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.855
train_ibs:  0.167


#### Test

In [58]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))
    
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [59]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=18, max_features=None, max_leaf_nodes=11,
                     max_samples=0.6101354912642183, min_samples_leaf=2,
                     min_weight_fraction_leaf=0.00292920588368519,
                     n_estimators=193, random_state=123, warm_start=True)

test_cindex:  0.601


RandomSurvivalForest(max_depth=10, max_features='auto', max_leaf_nodes=19,
                     max_samples=0.7880751381267705, min_samples_leaf=5,
                     min_samples_split=3,
                     min_weight_fraction_leaf=0.010141521629227908,
                     n_estimators=93, random_state=123)

test_ibs:  0.218


In [60]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [61]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

In [62]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-18 03:45:26,787] A new study created in memory with name: no-name-9f680ce1-eba5-4ad3-9562-031e47730219


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.7575757575757576
Fold 2 C-index: 0.8571428571428571
Fold 3 C-index: 0.8774509803921569
Fold 4 C-index: 0.8354430379746836
Fold 5 C-index: 0.6713615023474179
[I 2024-04-18 03:45:30,079] Trial 0 finished with value: 0.7997948270865746 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.7997948270865746.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-18 03:45:38,748] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531

Fold 5 C-index: 0.7018779342723005
[I 2024-04-18 03:47:12,591] Trial 15 finished with value: 0.802352653285175 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 9, 'min_samples_leaf': 5, 'max_depth': 9, 'n_estimators': 191, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.9428730581718685, 'min_weight_fraction_leaf': 0.10116500366378353}. Best is trial 15 with value: 0.802352653285175.
Fold 1 C-index: 0.7683982683982684
Fold 2 C-index: 0.8169642857142857
Fold 3 C-index: 0.8382352941176471
Fold 4 C-index: 0.8122362869198312
Fold 5 C-index: 0.6384976525821596
[I 2024-04-18 03:47:14,198] Trial 16 finished with value: 0.7748663575464383 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 17, 'min_samples_leaf': 8, 'max_depth': 9, 'n_estimators': 122, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.9902353348396513, 'min_weight_fraction_leaf': 0.4093818399278544}. Best is trial 15 with value: 0.80235265328517

Fold 1 C-index: 0.70995670995671
Fold 2 C-index: 0.8660714285714286
Fold 3 C-index: 0.8823529411764706
Fold 4 C-index: 0.8417721518987342
Fold 5 C-index: 0.7230046948356808
[I 2024-04-18 03:47:55,215] Trial 30 finished with value: 0.8046315852878049 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 15, 'min_samples_leaf': 3, 'max_depth': 5, 'n_estimators': 225, 'oob_score': False, 'warm_start': True, 'max_features': None, 'max_samples': 0.7156936401685365, 'min_weight_fraction_leaf': 0.07846912416680743}. Best is trial 30 with value: 0.8046315852878049.
Fold 1 C-index: 0.7142857142857143
Fold 2 C-index: 0.8660714285714286
Fold 3 C-index: 0.8823529411764706
Fold 4 C-index: 0.8417721518987342
Fold 5 C-index: 0.7230046948356808
[I 2024-04-18 03:47:57,953] Trial 31 finished with value: 0.8054973861536057 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 15, 'min_samples_leaf': 3, 'max_depth': 5, 'n_estimators': 225, 'oob_score': False, 'warm_start': True, 'max_features'

...

Fold 1 IBS: 0.1738388739675036
Fold 2 IBS: 0.1937875067454993
Fold 3 IBS: 0.17016113736228763
Fold 4 IBS: 0.17482337575841608
Fold 5 IBS: 0.21988648152779725
[I 2024-04-18 03:58:41,393] Trial 30 finished with value: 0.18649947507230077 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 11, 'min_samples_leaf': 9, 'max_depth': 8, 'n_estimators': 457, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.3838693489032564, 'min_weight_fraction_leaf': 0.03868084640768137}. Best is trial 12 with value: 0.17679008112418781.
Fold 1 IBS: 0.16823529544726903
Fold 2 IBS: 0.18935174649578138
Fold 3 IBS: 0.15723319929201915
Fold 4 IBS: 0.15395943956271183
Fold 5 IBS: 0.22622416365588105
[I 2024-04-18 03:58:56,543] Trial 31 finished with value: 0.1790007688907325 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 6, 'max_depth': 13, 'n_estimators': 408, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.

Fold 1 IBS: 0.16171040091818872
Fold 2 IBS: 0.1779507308726674
Fold 3 IBS: 0.1493687731616471
Fold 4 IBS: 0.14594036648959846
Fold 5 IBS: 0.2240383700080252
[I 2024-04-18 04:01:04,773] Trial 45 finished with value: 0.17180172829002538 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 5, 'min_samples_leaf': 2, 'max_depth': 18, 'n_estimators': 139, 'oob_score': False, 'warm_start': False, 'max_features': 'sqrt', 'max_samples': 0.5296598554254506, 'min_weight_fraction_leaf': 0.022410036988178828}. Best is trial 45 with value: 0.17180172829002538.
Fold 1 IBS: 0.1700981308082937
Fold 2 IBS: 0.18865396833856346
Fold 3 IBS: 0.16455531273130072
Fold 4 IBS: 0.1659095570486116
Fold 5 IBS: 0.22131350481069362
[I 2024-04-18 04:01:08,961] Trial 46 finished with value: 0.18210609474749262 and parameters: {'min_samples_split': 11, 'max_leaf_nodes': 7, 'min_samples_leaf': 2, 'max_depth': 17, 'n_estimators': 117, 'oob_score': False, 'warm_start': False, 'max_features': 'log2', 'max_samples': 0

Fold 1 IBS: 0.21398909457010623
Fold 2 IBS: 0.2207070361479251
Fold 3 IBS: 0.20528417601196522
Fold 4 IBS: 0.22484479482398192
Fold 5 IBS: 0.21754868037310363
[I 2024-04-18 04:02:51,810] Trial 60 finished with value: 0.21647475638541644 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 4, 'min_samples_leaf': 2, 'max_depth': 17, 'n_estimators': 344, 'oob_score': False, 'warm_start': False, 'max_features': 'sqrt', 'max_samples': 0.16250765016205834, 'min_weight_fraction_leaf': 0.1227005732829336}. Best is trial 45 with value: 0.17180172829002538.
Fold 1 IBS: 0.1642444466187744
Fold 2 IBS: 0.18371870561213746
Fold 3 IBS: 0.15486687513782893
Fold 4 IBS: 0.15128927567511535
Fold 5 IBS: 0.22489309120820142
[I 2024-04-18 04:03:01,753] Trial 61 finished with value: 0.1758024788504115 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 6, 'min_samples_leaf': 3, 'max_depth': 16, 'n_estimators': 305, 'oob_score': False, 'warm_start': False, 'max_features': 'log2', 'max_samples': 0

Fold 1 IBS: 0.16041964816322132
Fold 2 IBS: 0.18472776804985028
Fold 3 IBS: 0.15455347708913214
Fold 4 IBS: 0.1521283030894658
Fold 5 IBS: 0.22321385657424175
[I 2024-04-18 04:05:16,537] Trial 75 finished with value: 0.17500861059318226 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 4, 'min_samples_leaf': 2, 'max_depth': 14, 'n_estimators': 326, 'oob_score': False, 'warm_start': False, 'max_features': 'log2', 'max_samples': 0.2316710953741802, 'min_weight_fraction_leaf': 0.014344014173231634}. Best is trial 45 with value: 0.17180172829002538.
Fold 1 IBS: 0.19571061751658791
Fold 2 IBS: 0.20845642677508575
Fold 3 IBS: 0.18827483427372838
Fold 4 IBS: 0.20583148739593302
Fold 5 IBS: 0.21471598101862793
[I 2024-04-18 04:05:26,779] Trial 76 finished with value: 0.20259786939599259 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 1, 'max_depth': 15, 'n_estimators': 331, 'oob_score': False, 'warm_start': False, 'max_features': 0.1, 'max_samples': 0

Fold 1 IBS: 0.17277862124715318
Fold 2 IBS: 0.19027413414581626
Fold 3 IBS: 0.16904162603184877
Fold 4 IBS: 0.1735710939445359
Fold 5 IBS: 0.21703125661873038
[I 2024-04-18 04:08:16,750] Trial 90 finished with value: 0.18453934639761688 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 2, 'min_samples_leaf': 2, 'max_depth': 13, 'n_estimators': 372, 'oob_score': True, 'warm_start': False, 'max_features': 'sqrt', 'max_samples': 0.4950957244555919, 'min_weight_fraction_leaf': 0.02038605216364302}. Best is trial 45 with value: 0.17180172829002538.
Fold 1 IBS: 0.16656609819937618
Fold 2 IBS: 0.18778421598577213
Fold 3 IBS: 0.15585313805423273
Fold 4 IBS: 0.15189212933628532
Fold 5 IBS: 0.2232984092102556
[I 2024-04-18 04:08:30,282] Trial 91 finished with value: 0.17707879815718439 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 4, 'min_samples_leaf': 1, 'max_depth': 15, 'n_estimators': 353, 'oob_score': True, 'warm_start': False, 'max_features': 'log2', 'max_samples': 0.

In [63]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [64]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.83
train_ibs:  0.172


#### Test

In [65]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [66]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=4, max_features=None, max_leaf_nodes=13,
                   max_samples=0.6016058414925255, min_samples_leaf=2,
                   min_samples_split=9,
                   min_weight_fraction_leaf=0.012524279113161611,
                   n_estimators=79, oob_score=True, random_state=123,
                   warm_start=True)

C-index score: 0.629


ExtraSurvivalTrees(max_depth=18, max_leaf_nodes=5,
                   max_samples=0.5296598554254506, min_samples_leaf=2,
                   min_samples_split=8,
                   min_weight_fraction_leaf=0.022410036988178828,
                   n_estimators=139, random_state=123)

IBS: 0.213


In [67]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis


#### Train

In [68]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-18 04:10:28,933] A new study created in memory with name: no-name-d3036d4f-db00-4649-96ee-3df31035c35f


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-18 04:11:34,777] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-18 04:12:09,881] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-18 04:31:10,696] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.8386796524426539, 'learning_rate': 0.046734492485875676, 'dropout_rate': 0.4821375662037144, 'n_estimators': 402, 'criterion': 'squared_error', 'ccp_alpha': 1.5696007313501796, 'min_weight_fraction_leaf': 0.18684147934268416, 'max_features': 'log2', 'min_impurity_decrease': 1.5044881127471587e-06, 'validation_fraction': 0.8166356053932342, 'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 16, 'max_depth': 4}. Best is trial 9 with value: 0.75875287105312.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-18 04:33:29,698] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.33235389014851724, 'learning_rate': 0.04522573411670834, 'dropout_rate': 0.2712811374536856, 'n_estimators': 405, 'criterion': 'squared_

Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-18 04:58:23,063] Trial 25 finished with value: 0.5 and parameters: {'subsample': 0.7565257046780437, 'learning_rate': 0.01188344684416992, 'dropout_rate': 0.3433523077110169, 'n_estimators': 318, 'criterion': 'squared_error', 'ccp_alpha': 2.063559212730723, 'min_weight_fraction_leaf': 0.25401273903421573, 'max_features': 'auto', 'min_impurity_decrease': 5.773435946665558e-07, 'validation_fraction': 0.8062268184400477, 'min_samples_split': 16, 'max_leaf_nodes': 18, 'min_samples_leaf': 5, 'max_depth': 3}. Best is trial 9 with value: 0.75875287105312.
Fold 1 C-index: 0.6753246753246753
Fold 2 C-index: 0.6004464285714286
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.6103286384976526
[I 2024-04-18 05:01:18,783] Trial 26 finished with value: 0.5772199484787512 and parameters: {'subsample': 0.8922404482621683, 'learning_rate': 0.011585260292674536, 'dropout_rate': 0.2527000999648632, 'n_estimators': 446, 'crite

...

Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609557
[I 2024-04-18 07:35:58,509] Trial 44 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.9992870370700113, 'learning_rate': 0.022847552015173876, 'dropout_rate': 0.1556807870961761, 'n_estimators': 451, 'criterion': 'squared_error', 'ccp_alpha': 1.6646055539220843, 'min_weight_fraction_leaf': 0.1403134453903068, 'max_features': 'auto', 'min_impurity_decrease': 2.7552659293421345e-07, 'validation_fraction': 0.8820166309308186, 'min_samples_split': 17, 'max_leaf_nodes': 14, 'min_samples_leaf': 10, 'max_depth': 2}. Best is trial 42 with value: 0.21333440679035767.
Fold 1 IBS: 0.21266963391512192
Fold 2 IBS: 0.220226363872871
Fold 3 IBS: 0.20319998528341784
Fold 4 IBS: 0.22260228978618746
Fold 5 IBS: 0.21744807253949516
[I 2024-04-18 07:37:12,247] Trial 45 finished with value: 0.2152292690794187 and parameters: {'subsample': 0.8888212863898438, 'learning_rate': 0.015249832110

Fold 3 IBS: 0.20287635124596737
Fold 4 IBS: 0.22157930195318806
Fold 5 IBS: 0.21757113828505154
[I 2024-04-18 07:53:44,855] Trial 55 finished with value: 0.21482590116759398 and parameters: {'subsample': 0.9703353679292269, 'learning_rate': 0.023646082998228058, 'dropout_rate': 0.2676210325613897, 'n_estimators': 481, 'criterion': 'squared_error', 'ccp_alpha': 0.036860238643527846, 'min_weight_fraction_leaf': 0.21798842867076448, 'max_features': None, 'min_impurity_decrease': 4.086647023052284e-07, 'validation_fraction': 0.8868940629916056, 'min_samples_split': 18, 'max_leaf_nodes': 15, 'min_samples_leaf': 10, 'max_depth': 4}. Best is trial 42 with value: 0.21333440679035767.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609582
[I 2024-04-18 07:55:48,588] Trial 56 finished with value: 0.21659054862241592 and parameters: {'subsample': 0.9556274374724505, 'learning_rate': 0.022777236

Fold 1 IBS: 0.21397652195149616
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.2181243152560957
[I 2024-04-18 08:01:49,990] Trial 67 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.9699873838014033, 'learning_rate': 0.012103946738880062, 'dropout_rate': 0.14222380728521514, 'n_estimators': 115, 'criterion': 'squared_error', 'ccp_alpha': 0.29393333981111347, 'min_weight_fraction_leaf': 0.030567908908384282, 'max_features': 'sqrt', 'min_impurity_decrease': 1.094611645458025e-07, 'validation_fraction': 0.8187342892372662, 'min_samples_split': 17, 'max_leaf_nodes': 12, 'min_samples_leaf': 10, 'max_depth': 12}. Best is trial 42 with value: 0.21333440679035767.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609563
[I 2024-04-18 08:01:50,969] Trial 68 finished with value: 0.21659054862241586 and paramete

Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018132
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.2181243152560957
[I 2024-04-18 08:02:22,759] Trial 78 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.9362147727385395, 'learning_rate': 0.02736082926046323, 'dropout_rate': 0.26541734467806316, 'n_estimators': 77, 'criterion': 'friedman_mse', 'ccp_alpha': 0.2910342652485194, 'min_weight_fraction_leaf': 0.22600697495173297, 'max_features': 'log2', 'min_impurity_decrease': 2.73345967920419e-07, 'validation_fraction': 0.9235701874070005, 'min_samples_split': 16, 'max_leaf_nodes': 17, 'min_samples_leaf': 11, 'max_depth': 4}. Best is trial 42 with value: 0.21333440679035767.
Fold 1 IBS: 0.21397652195149616
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609574
[I 2024-04-18 08:02:29,382] Trial 79 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.9100835469049136, '

Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609566
[I 2024-04-18 08:03:24,338] Trial 89 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.9828049683854578, 'learning_rate': 0.0401442321523931, 'dropout_rate': 0.12508897364625737, 'n_estimators': 132, 'criterion': 'friedman_mse', 'ccp_alpha': 1.1917037663435968, 'min_weight_fraction_leaf': 0.1418992381857624, 'max_features': 'log2', 'min_impurity_decrease': 4.784822523408194e-07, 'validation_fraction': 0.9230642450597242, 'min_samples_split': 19, 'max_leaf_nodes': 18, 'min_samples_leaf': 14, 'max_depth': 9}. Best is trial 42 with value: 0.21333440679035767.
Fold 1 IBS: 0.21397652195149616
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018132
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.2181243152560958
[I 2024-04-18 08:03:25,429] Trial 90 finished with value: 0.21659054862241592 and parameters: {'subsample': 0.7850349664677921, 'learning_rate': 0.027950629726

In [69]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [70]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.791
train_ibs:  0.213


#### Test

In [71]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [72]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.008819927083844869,
                                 criterion='squared_error',
                                 dropout_rate=0.6757649295445344,
                                 learning_rate=0.023239756862861477,
                                 max_depth=1, max_features='sqrt',
                                 max_leaf_nodes=16,
                                 min_impurity_decrease=0.0021237516252394237,
                                 min_samples_leaf=18, min_samples_split=17,
                                 min_weight_fraction_leaf=0.1942611713172168,
                                 n_estimators=53, random_state=123,
                                 subsample=0.6001028949487068,
                                 validation_fraction=0.7847712019774569)

C-index score: 0.647


GradientBoostingSurvivalAnalysis(ccp_alpha=0.008327408119788757,
                                 criterion='squared_error',
                                 dropout_rate=0.10802959125648778,
                                 learning_rate=0.012507391626216524,
                                 max_depth=2, max_features='auto',
                                 max_leaf_nodes=16,
                                 min_impurity_decrease=1.0593992090305253e-07,
                                 min_samples_leaf=13, min_samples_split=19,
                                 min_weight_fraction_leaf=0.13510597987400375,
                                 n_estimators=481, random_state=123,
                                 subsample=0.9646924257252548,
                                 validation_fraction=0.9483244851205971)

IBS: 0.219


In [73]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [74]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

In [75]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-18 08:06:40,446] A new study created in memory with name: no-name-df9e9486-c7a3-49de-995d-7e9e5893d061


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.7633928571428571
Fold 3 C-index: 0.7156862745098039
Fold 4 C-index: 0.6962025316455697
Fold 5 C-index: 0.6713615023474179
[I 2024-04-18 08:06:41,643] Trial 0 finished with value: 0.6818827456832424 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.6818827456832424.
Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.7633928571428571
Fold 3 C-index: 0.7156862745098039
Fold 4 C-index: 0.6962025316455697
Fold 5 C-index: 0.6713615023474179
[I 2024-04-18 08:06:51,502] Trial 1 finished with value: 0.6818827456832424 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 0 with value: 0.6818827456832424.
Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.7633928571428571
Fold 3 C-index: 0.7156862745098039
Fold 

Fold 1 C-index: 0.5670995670995671
Fold 2 C-index: 0.8035714285714286
Fold 3 C-index: 0.7254901960784313
Fold 4 C-index: 0.70042194092827
Fold 5 C-index: 0.6854460093896714
[I 2024-04-18 08:08:19,734] Trial 19 finished with value: 0.6964058284134736 and parameters: {'subsample': 0.10013306718236009, 'dropout_rate': 0.7254192287154788, 'n_estimators': 117, 'learning_rate': 0.09614402133777997}. Best is trial 12 with value: 0.7116176878051177.
Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.7857142857142857
Fold 3 C-index: 0.7205882352941176
Fold 4 C-index: 0.6962025316455697
Fold 5 C-index: 0.6666666666666666
[I 2024-04-18 08:08:29,177] Trial 20 finished with value: 0.6863884564182405 and parameters: {'subsample': 0.2630057481431337, 'dropout_rate': 0.18040218016888274, 'n_estimators': 423, 'learning_rate': 0.07788582119853761}. Best is trial 12 with value: 0.7116176878051177.
Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.8035714285714286
Fold 3 C-index: 0.7254901960784313
F

Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.8080357142857143
Fold 3 C-index: 0.7401960784313726
Fold 4 C-index: 0.70042194092827
Fold 5 C-index: 0.6807511737089202
[I 2024-04-18 08:09:50,447] Trial 38 finished with value: 0.698435094024968 and parameters: {'subsample': 0.10130603260320498, 'dropout_rate': 0.9265417979312904, 'n_estimators': 154, 'learning_rate': 0.06034311549365794}. Best is trial 12 with value: 0.7116176878051177.
Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.7633928571428571
Fold 3 C-index: 0.7156862745098039
Fold 4 C-index: 0.6962025316455697
Fold 5 C-index: 0.6713615023474179
[I 2024-04-18 08:09:53,499] Trial 39 finished with value: 0.6818827456832424 and parameters: {'subsample': 0.6100491404460081, 'dropout_rate': 0.6000657611403288, 'n_estimators': 233, 'learning_rate': 0.08287173573083674}. Best is trial 12 with value: 0.7116176878051177.
Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.7723214285714286
Fold 3 C-index: 0.7156862745098039
Fol

...

Fold 2 IBS: 0.1806549522812558
Fold 3 IBS: 0.2626642449139272
Fold 4 IBS: 0.22142266311808342
Fold 5 IBS: 0.21718912459536338
[I 2024-04-18 08:15:29,453] Trial 19 finished with value: 0.23815290611190704 and parameters: {'subsample': 0.38793667853472646, 'dropout_rate': 0.22827203020188386, 'n_estimators': 324, 'learning_rate': 0.08697733590757546}. Best is trial 6 with value: 0.17708013385298887.
Fold 1 IBS: 0.21571820396955296
Fold 2 IBS: 0.1504945945870415
Fold 3 IBS: 0.1720434961065403
Fold 4 IBS: 0.1707499999148482
Fold 5 IBS: 0.1731189833324429
[I 2024-04-18 08:15:31,399] Trial 20 finished with value: 0.17642505558208518 and parameters: {'subsample': 0.6102628884762309, 'dropout_rate': 0.5558690358940764, 'n_estimators': 149, 'learning_rate': 0.017331004877487094}. Best is trial 20 with value: 0.17642505558208518.
Fold 1 IBS: 0.21422382295710835
Fold 2 IBS: 0.15300561222318346
Fold 3 IBS: 0.17291484540313803
Fold 4 IBS: 0.17228920130049374
Fold 5 IBS: 0.17394906275993324
[I 2024-

Fold 2 IBS: 0.14792185158467971
Fold 3 IBS: 0.21099096752400537
Fold 4 IBS: 0.17749683277884493
Fold 5 IBS: 0.18951087472922412
[I 2024-04-18 08:16:33,263] Trial 38 finished with value: 0.19751383100272865 and parameters: {'subsample': 0.672433369611978, 'dropout_rate': 0.8916907706526339, 'n_estimators': 406, 'learning_rate': 0.02207087345824238}. Best is trial 36 with value: 0.17545126273803033.
Fold 1 IBS: 0.23420785619175652
Fold 2 IBS: 0.13885849523121738
Fold 3 IBS: 0.17671673311532535
Fold 4 IBS: 0.16499092522728498
Fold 5 IBS: 0.173610528604103
[I 2024-04-18 08:16:37,581] Trial 39 finished with value: 0.17767690767393746 and parameters: {'subsample': 0.44350783452649856, 'dropout_rate': 0.8560401235471927, 'n_estimators': 337, 'learning_rate': 0.013270934894443748}. Best is trial 36 with value: 0.17545126273803033.
Fold 1 IBS: 0.205387901723201
Fold 2 IBS: 0.18350743788057733
Fold 3 IBS: 0.1849715526954976
Fold 4 IBS: 0.19513263070538195
Fold 5 IBS: 0.19071458823853912
[I 2024-

Fold 2 IBS: 0.14136069617897423
Fold 3 IBS: 0.19077540790490488
Fold 4 IBS: 0.1685716554770952
Fold 5 IBS: 0.18079976141115142
[I 2024-04-18 08:17:24,578] Trial 57 finished with value: 0.18588459850909891 and parameters: {'subsample': 0.722713662188781, 'dropout_rate': 0.910417632083181, 'n_estimators': 310, 'learning_rate': 0.020570994628935912}. Best is trial 36 with value: 0.17545126273803033.
Fold 1 IBS: 0.21269836118729812
Fold 2 IBS: 0.15512924316813698
Fold 3 IBS: 0.17363497274043274
Fold 4 IBS: 0.17400050761889904
Fold 5 IBS: 0.17502156774388983
[I 2024-04-18 08:17:26,458] Trial 58 finished with value: 0.17809693049173134 and parameters: {'subsample': 0.6249882398313333, 'dropout_rate': 0.9713770012669294, 'n_estimators': 184, 'learning_rate': 0.012229054440115003}. Best is trial 36 with value: 0.17545126273803033.
Fold 1 IBS: 0.20550726620803447
Fold 2 IBS: 0.17740120082704586
Fold 3 IBS: 0.18179949438193768
Fold 4 IBS: 0.1901135316676914
Fold 5 IBS: 0.1871967533189637
[I 2024

Fold 2 IBS: 0.13955997775414206
Fold 3 IBS: 0.1807687133909246
Fold 4 IBS: 0.16556961812830692
Fold 5 IBS: 0.175676217950483
[I 2024-04-18 08:18:19,747] Trial 76 finished with value: 0.18011051892898547 and parameters: {'subsample': 0.6018652093062146, 'dropout_rate': 0.8548910278043131, 'n_estimators': 209, 'learning_rate': 0.024235900006421195}. Best is trial 36 with value: 0.17545126273803033.
Fold 1 IBS: 0.20536765150306427
Fold 2 IBS: 0.18014952635091636
Fold 3 IBS: 0.183023785063995
Fold 4 IBS: 0.19223726439760763
Fold 5 IBS: 0.18830661932475276
[I 2024-04-18 08:18:21,050] Trial 77 finished with value: 0.1898169693280672 and parameters: {'subsample': 0.6241144322095822, 'dropout_rate': 0.8105208156539707, 'n_estimators': 141, 'learning_rate': 0.008359784620781453}. Best is trial 36 with value: 0.17545126273803033.
Fold 1 IBS: 0.2374025235368971
Fold 2 IBS: 0.13948712259061768
Fold 3 IBS: 0.1789264161523359
Fold 4 IBS: 0.16528306816225447
Fold 5 IBS: 0.1748234198685515
[I 2024-04-

Fold 2 IBS: 0.16478959838439622
Fold 3 IBS: 0.17653755878235672
Fold 4 IBS: 0.1805950354677507
Fold 5 IBS: 0.17998076390593717
[I 2024-04-18 08:19:24,893] Trial 95 finished with value: 0.18198923228208327 and parameters: {'subsample': 0.5198280584984859, 'dropout_rate': 0.8486478738783642, 'n_estimators': 294, 'learning_rate': 0.006037141349276085}. Best is trial 36 with value: 0.17545126273803033.
Fold 1 IBS: 0.23515216169778982
Fold 2 IBS: 0.13954617558808188
Fold 3 IBS: 0.17705832296971719
Fold 4 IBS: 0.16501460559381048
Fold 5 IBS: 0.17386940040131596
[I 2024-04-18 08:19:28,193] Trial 96 finished with value: 0.17812813325014307 and parameters: {'subsample': 0.6271111992646494, 'dropout_rate': 0.892295025185902, 'n_estimators': 249, 'learning_rate': 0.018198796414258485}. Best is trial 36 with value: 0.17545126273803033.
Fold 1 IBS: 0.21259560893410287
Fold 2 IBS: 0.15514525884910346
Fold 3 IBS: 0.17326490031645939
Fold 4 IBS: 0.1738706026163926
Fold 5 IBS: 0.1751375875495752
[I 202

In [76]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [77]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.712
train_ibs:  0.175


#### Test

In [78]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [79]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.10707700162921632,
                                              learning_rate=0.09635955935176935,
                                              n_estimators=311,
                                              random_state=123,
                                              subsample=0.11211713718471546)

C-index score: 0.598


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.7803451305864744,
                                              learning_rate=0.012746071877375342,
                                              n_estimators=251,
                                              random_state=123,
                                              subsample=0.6689949239769296)

IBS: 0.231


In [80]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

## Results

In [81]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
Randomsurvivalforest,0.855,1.0
ExtraSurvivalTrees,0.830,2.0
GradientBoosting,0.791,3.0
CoxElastic,0.780,4.0
CoxLasso,0.779,5.0
CoxPH,0.778,6.0
ComponentwiseGradientBoosting,0.712,7.0
CoxRidge,0.681,8.0


In [82]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
Randomsurvivalforest,0.167,1.0
CoxElastic,0.168,2.0
CoxPH,0.169,3.5
CoxLasso,0.169,3.5
ExtraSurvivalTrees,0.172,5.0
ComponentwiseGradientBoosting,0.175,6.0
GradientBoosting,0.213,7.0
CoxRidge,0.217,8.0


In [83]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
GradientBoosting,0.647,1.0
ExtraSurvivalTrees,0.629,2.0
CoxLasso,0.623,3.5
CoxElastic,0.623,3.5
CoxPH,0.620,5.0
Randomsurvivalforest,0.601,6.0
ComponentwiseGradientBoosting,0.598,7.0
CoxRidge,0.578,8.0


In [84]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs 

,IBS,rank
ExtraSurvivalTrees,0.213,1.0
Randomsurvivalforest,0.218,2.0
GradientBoosting,0.219,3.0
CoxRidge,0.221,4.0
CoxElastic,0.229,5.0
CoxLasso,0.230,6.0
ComponentwiseGradientBoosting,0.231,7.0
CoxPH,0.232,8.0


In [85]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  
file_path = 'path_to_your_folder/'  

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d3/os/standard/plsr/' 

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

# Modify the file names to match the desired format
modified_file_names = ['d3_os_standard_plsr_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [86]:
from datetime import date

current_date = date.today()
print(current_date)

2024-04-18
